# Bài 8.2 · Phân tích dữ liệu trước huấn luyện

**Câu hỏi của buổi này:** trước khi chọn model, nhìn vào dữ liệu những gì để điểm validation nói thật, và mỗi điều thấy được dẫn tới việc gì?

Hai bộ dữ liệu thật của UCI:

- **Bike Sharing**: số lượt thuê xe đạp mỗi giờ ở Washington D.C., 2011–2012, kèm thời tiết. Bài toán: đoán số lượt.
- **SMS Spam**: 5 574 tin nhắn tiếng Anh có nhãn spam hoặc không.

| Mục | Sẽ thấy |
|---|---|
| 1. Đọc file | bảng cột, giờ bị thiếu, lượt thuê theo ngày và theo giờ |
| 2. Mẫu trùng | tin nhắn trùng hệt và gần trùng; điểm trên câu đã gặp và câu mới |
| 3. Validation giống lúc dùng | cùng một model, ba cách chia, ba con số |
| 4. Rò rỉ | một cột cộng lại ra đúng nhãn |
| 5. Lệch phân phối | học năm 2011, đoán năm 2012; adversarial validation |
| 6. Ablation | bỏ từng nhóm cột, điểm đổi bao nhiêu |
| 7. Lát dữ liệu | sai số theo giờ và theo thời tiết |
| 8. Tóm tắt | một bảng các con số chính |

Các dòng ✓ là phép kiểm in ra: chạy lại mà thấy ✗ thì có gì đó sai. Chỉ cần CPU, bật **Internet** trong *Session options*; chạy hết khoảng 3 phút. Bài giảng: [ai-canuc80k · bài 8.2](https://ai-canuc80k.ai-web-deploy-20261010.workers.dev/#/lesson/8.2).

In [ ]:
# Ô tiện ích: định dạng bảng, dòng kiểm ✓/✗, nhãn số trên biểu đồ. Không cần đọc để theo bài.
import warnings, contextlib, io, logging
warnings.filterwarnings('ignore')
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from IPython.display import display, HTML

plt.rcParams.update({
    'figure.dpi': 110, 'figure.figsize': (8, 4), 'font.size': 11, 'axes.titlesize': 12, 'axes.titleweight': 'bold',
    'axes.spines.top': False, 'axes.spines.right': False, 'axes.grid': True, 'grid.alpha': .25,
    'legend.frameon': False, 'image.cmap': 'viridis',
})
C = {'blue': '#3b6ea8', 'clay': '#c2603e', 'sand': '#b8892b', 'green': '#4d7337', 'gray': '#8a877f'}
ORDER = [C['blue'], C['clay'], C['sand'], C['green'], C['gray']]
plt.rcParams['axes.prop_cycle'] = plt.cycler(color=ORDER)


def table(df, title=None, fmt=None, best=None, lower_is_better=False, index=False):
    """In một bảng gọn: định dạng số, tô đậm ô tốt nhất của cột `best`."""
    st = df.style.format(fmt, na_rep='–')
    if best is not None:
        col = df[best]
        target = col.min() if lower_is_better else col.max()
        st = st.apply(lambda s: ['font-weight:700;background:#f5e6dd' if (s.name == best and v == target) else '' for v in s], axis=0)
    text_cols = [c for c in df.columns if not pd.api.types.is_numeric_dtype(df[c])]
    if text_cols:
        st = st.set_properties(subset=text_cols, **{'text-align': 'left'})
    if not index:
        st = st.hide(axis='index')
    st = st.set_table_styles([
        {'selector': 'th', 'props': 'text-align:left;padding:6px 12px;border-bottom:2px solid #d3cebf;font-weight:600'},
        {'selector': 'td', 'props': 'padding:5px 12px;border-bottom:1px solid #e6e2d6'},
        {'selector': 'caption', 'props': 'caption-side:top;text-align:left;font-weight:700;font-size:14px;padding:4px 0 8px;white-space:nowrap'},
    ])
    if title:
        st = st.set_caption(title)
    display(st)


def check(msg, ok):
    """Dòng kiểm tra in ra: ✓ khi đúng, ✗ khi sai (không dừng notebook)."""
    print(('✓ ' if ok else '✗ ') + msg)


def bar_labels(ax, fmt='{:.2f}'):
    """Ghi giá trị lên đầu mỗi thanh của biểu đồ cột."""
    for cont in ax.containers:
        ax.bar_label(cont, labels=[fmt.format(v) for v in cont.datavalues], padding=3, fontsize=9)


@contextlib.contextmanager
def quiet():
    """Ẩn thanh tiến trình và dòng thông báo khi tải dữ liệu hay trọng số."""
    logging.disable(logging.WARNING)
    try:
        with contextlib.redirect_stdout(io.StringIO()), contextlib.redirect_stderr(io.StringIO()):
            yield
    finally:
        logging.disable(logging.NOTSET)


def note(text):
    """Một dòng nhận xét in đậm nhẹ dưới output, tính từ số thật của lần chạy."""
    display(HTML(f'<div style="border-left:3px solid #c2603e;padding:4px 10px;margin:6px 0;color:#444">{text}</div>'))

In [ ]:
import io, re, time, zipfile, urllib.request
from pathlib import Path
import sklearn
from sklearn.ensemble import HistGradientBoostingRegressor, HistGradientBoostingClassifier
from sklearn.model_selection import KFold, GroupKFold, cross_val_predict, train_test_split
from sklearn.metrics import mean_squared_error, r2_score, roc_auc_score, f1_score
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.pipeline import make_pipeline

table(pd.DataFrame([('scikit-learn', sklearn.__version__), ('pandas', pd.__version__)], columns=['thành phần', 'phiên bản']), 'Môi trường chạy')

def fetch(url, folder):
    root = Path('/tmp') / folder; root.mkdir(parents=True, exist_ok=True)
    if not (root / 'data.zip').exists(): urllib.request.urlretrieve(url, root / 'data.zip')
    return zipfile.ZipFile(root / 'data.zip')
bike_zip = fetch('https://archive.ics.uci.edu/static/public/275/bike+sharing+dataset.zip', 'bike')
sms_zip = fetch('https://archive.ics.uci.edu/static/public/228/sms+spam+collection.zip', 'sms')

def rmse(a, b): return float(np.sqrt(mean_squared_error(a, b)))
def gbm(): return HistGradientBoostingRegressor(max_iter=300, random_state=0)

## 1. Đọc file trước khi đọc model

Mở `hour.csv` của Bike Sharing: mỗi dòng là một giờ. Đọc kèm `Readme.txt`, vì nhiều cột đã bị biến đổi: `temp` đã chia cho 41, `hum` chia cho 100, `weathersit` là mã 1–4 (1 trời quang, 4 mưa to).

In [ ]:
h = pd.read_csv(bike_zip.open('hour.csv'), parse_dates=['dteday'])
readme = bike_zip.read('Readme.txt').decode('latin-1')
cols = pd.DataFrame({'cột': h.columns, 'kiểu': h.dtypes.astype(str).values, 'số giá trị khác nhau': h.nunique().values,
                     'ví dụ': [str(h[c].iloc[100]) for c in h.columns], 'ô trống': h.isna().sum().values})
table(cols, f'hour.csv: {len(h):,} dòng × {h.shape[1]} cột'.replace(',', ' '))

days = h['dteday'].nunique()
per_day = h.groupby('dteday').size()
full = pd.MultiIndex.from_product([h['dteday'].unique(), range(24)], names=['dteday', 'hr'])
missing_hr = full.difference(pd.MultiIndex.from_frame(h[['dteday', 'hr']])).get_level_values('hr')
night = np.isin(missing_hr, range(0, 6)).mean()
miss = pd.DataFrame([('số ngày', days), ('số giờ phải có (ngày × 24)', days * 24), ('số dòng thật có', len(h)),
                     ('giờ bị thiếu', days * 24 - len(h)), ('ngày thiếu giờ', int((per_day < 24).sum())),
                     ('ngày thiếu nhiều nhất', f'{per_day.idxmin():%d/%m/%Y}: còn {per_day.min()} giờ'),
                     ('số dòng weathersit = 4 (mưa to)', int((h['weathersit'] == 4).sum()))], columns=['', 'giá trị'])
table(miss, 'Giờ bị thiếu không hiện thành ô trống: nó là dòng không tồn tại')
check('không có ô trống nào, nhưng vẫn thiếu giờ', h.isna().sum().sum() == 0 and len(h) < days * 24)
check('Readme ghi temp đã chia cho 41', '41' in readme and 'temp' in readme)

fig, axes = plt.subplots(1, 2, figsize=(15, 4.2), gridspec_kw={'width_ratios': [1.6, 1]})
daily = h.groupby('dteday')['cnt'].sum()
axes[0].plot(daily.index, daily.values, color=C['blue'], lw=.8)
axes[0].plot(daily.rolling(30, center=True).mean(), color=C['clay'], lw=2, label='trung bình 30 ngày')
axes[0].set(ylabel='lượt thuê mỗi ngày', title='Hai năm: theo mùa, và năm sau đông hơn năm trước'); axes[0].legend(frameon=False)
piv = h.pivot_table(index='weekday', columns='hr', values='cnt', aggfunc='mean').reindex([1, 2, 3, 4, 5, 6, 0])
im = axes[1].imshow(piv.values, aspect='auto', cmap='magma')
axes[1].set_yticks(range(7), ['T2', 'T3', 'T4', 'T5', 'T6', 'T7', 'CN']); axes[1].set_xticks(range(0, 24, 3), range(0, 24, 3))
axes[1].set(xlabel='giờ trong ngày', title='Lượt thuê trung bình theo thứ và giờ'); fig.colorbar(im, ax=axes[1], fraction=.04)
plt.tight_layout(); plt.show()
wd = piv.loc[[1, 2, 3, 4, 5]].mean()
peak = [int(wd[:12].idxmax()), int(wd[12:].idxmax())]     # đỉnh buổi sáng, đỉnh buổi chiều
note(f'Không ô nào trống, nhưng {days * 24 - len(h)} giờ không có dòng ({night:.0%} trong số đó rơi vào 0–5 giờ sáng, lúc không ai thuê nên không có bản ghi): đếm theo ngày mới thấy. '
     f'Ngày thường có hai đỉnh lúc {peak[0]} giờ và {peak[1]} giờ (đi làm, tan làm), cuối tuần đông vào buổi chiều: '
     f'giờ và loại ngày sẽ là hai cột quan trọng. Năm 2012 đông hơn năm 2011 rõ rệt; mục 5 quay lại chuyện này.')

## 2. Mẫu trùng không chỉ là hai hàng giống hệt

Sang SMS Spam. Đếm tin trùng hệt, rồi tin gần trùng: viết thường, đổi mọi chữ số thành `0`, bỏ dấu câu. Nếu một tin có bản sao nằm cả ở train lẫn validation, model chỉ cần nhớ là đúng.

In [ ]:
raw = sms_zip.read('SMSSpamCollection').decode('utf-8', 'replace').strip().split('\n')
sms = pd.DataFrame([l.split('\t', 1) for l in raw], columns=['nhãn', 'text'])
y_sms = (sms['nhãn'] == 'spam').astype(int).values
norm = sms['text'].str.lower().str.replace(r'\d', '0', regex=True).str.replace(r'[^a-z0 ]', ' ', regex=True).str.split().str.join(' ')
norm = norm.where(norm.str.len() > 0, sms['text'])     # tin chỉ có ký hiệu như ':)' thì giữ nguyên
conflict = int((sms.groupby('text')['nhãn'].nunique() > 1).sum())
dup = pd.DataFrame([('số tin', len(sms)), ('spam', f'{y_sms.mean():.1%}'), ('tin trùng hệt một tin trước đó', int(sms['text'].duplicated().sum())),
                    ('tin gần trùng (sau khi chuẩn hóa)', int(norm.duplicated().sum())), ('cùng chữ mà khác nhãn', conflict)], columns=['', 'giá trị'])
table(dup, 'SMS Spam')
top = sms['text'].value_counts().head(4)
variants = sms.groupby(norm)['text'].unique()
pair = list(variants[(variants.map(len) >= 2) & (variants.index.str.len() >= 25)].iloc[0][:2])     # hai cách viết khác nhau của cùng một tin
table(pd.DataFrame({'tin': list(top.index) + pair, 'số lần': list(top.values) + ['gần trùng', 'gần trùng']}), 'Tin lặp nhiều nhất, và một cặp chỉ khác chữ số hoặc dấu câu')

def memorizer(): return make_pipeline(TfidfVectorizer(), KNeighborsClassifier(1, metric='cosine'))
def linear(): return make_pipeline(TfidfVectorizer(), LogisticRegression(C=10, max_iter=1000))
rows = []
for name, mk in [('1-NN (chép tin gần nhất)', memorizer), ('logistic trên TF-IDF', linear)]:
    acc_seen, acc_new, share = [], [], []
    for seed in range(5):
        tr, va = train_test_split(np.arange(len(sms)), test_size=.2, random_state=seed, stratify=y_sms)
        p = mk().fit(sms['text'][tr], y_sms[tr]).predict(sms['text'][va])
        seen = sms['text'][va].isin(set(sms['text'][tr])).values
        acc_seen.append((p[seen] == y_sms[va][seen]).mean()); acc_new.append((p[~seen] == y_sms[va][~seen]).mean()); share.append(seen.mean())
    f_rand = f1_score(y_sms, cross_val_predict(mk(), sms['text'], y_sms, cv=KFold(5, shuffle=True, random_state=0)))
    f_group = f1_score(y_sms, cross_val_predict(mk(), sms['text'], y_sms, cv=GroupKFold(5), groups=pd.factorize(norm)[0]))
    rows.append((name, np.mean(share), np.mean(acc_seen), np.mean(acc_new), f_rand, f_group))
mem = pd.DataFrame(rows, columns=['model', 'tin validation đã có trong train', 'accuracy: tin đã gặp', 'accuracy: tin mới', 'F1 spam: chia ngẫu nhiên', 'F1 spam: nhóm tin gần trùng'])
table(mem, 'Chia ngẫu nhiên 80/20 (trung bình 5 lần), và 5-fold theo hai cách', fmt={c: '{:.3f}' for c in mem.columns[1:]} | {mem.columns[1]: '{:.0%}'})
check('có tin trùng hệt và không có tin cùng chữ khác nhãn', dup.iloc[2, 1] > 0 and conflict == 0)
check('tin đã gặp trong train được đoán đúng nhiều hơn tin mới', (mem['accuracy: tin đã gặp'] > mem['accuracy: tin mới']).all())
r0 = mem.iloc[0]
note(f'Khoảng {r0.iloc[1]:.0%} tin validation đã có bản y hệt trong train; với chúng 1-NN đúng {r0.iloc[2]:.1%}, với tin mới chỉ {r0.iloc[3]:.1%}. '
     f'Gom mọi tin gần trùng vào cùng một fold, F1 của 1-NN đổi từ {r0.iloc[4]:.3f} thành {r0.iloc[5]:.3f}. '
     'Ở đây tin trùng không mâu thuẫn nhãn nên chênh lệch nhỏ; với ảnh chụp liên tiếp hay tài liệu sao chép, phần "đã gặp" có thể chiếm phần lớn validation.')

## 3. Validation phải giống câu hỏi lúc sử dụng

Model đoán lượt thuê sẽ được dùng cho **những ngày sắp tới**. Thử ba cách chia trên cùng model (gradient boosting, nhãn là log(1 + số lượt), chấm bằng RMSE):

- **chia ngẫu nhiên theo giờ**: giờ 8 và giờ 9 của cùng một ngày có thể rơi vào hai phía;
- **chia theo ngày** (GroupKFold): mọi giờ của một ngày nằm cùng một phía;
- **chia theo thời gian**: học trước 1/7/2012, chấm 6 tháng sau đó, đúng như lúc dùng thật.

In [ ]:
FEAT = ['season', 'yr', 'mnth', 'hr', 'holiday', 'weekday', 'workingday', 'weathersit', 'temp', 'atemp', 'hum', 'windspeed']
y = np.log1p(h['cnt'])
past, future = h['dteday'] < '2012-07-01', h['dteday'] >= '2012-07-01'
hp, yp = h[past].reset_index(drop=True), y[past].reset_index(drop=True)
t = time.time()
r_rand = rmse(yp, cross_val_predict(gbm(), hp[FEAT], yp, cv=KFold(5, shuffle=True, random_state=0)))
r_day = rmse(yp, cross_val_predict(gbm(), hp[FEAT], yp, cv=GroupKFold(5), groups=hp['dteday']))
model_time = gbm().fit(h.loc[past, FEAT], y[past])
pred_future = model_time.predict(h.loc[future, FEAT])
r_time = rmse(y[future], pred_future)
sp = pd.DataFrame([('ngẫu nhiên theo giờ (5-fold, 18 tháng đầu)', r_rand), ('theo ngày (5-fold, 18 tháng đầu)', r_day), ('theo thời gian (học 18 tháng, chấm 6 tháng sau)', r_time)],
                  columns=['cách chia', 'RMSE trên log(1 + lượt)'])
fig, ax = plt.subplots(figsize=(10, 3.2))
ax.barh(sp['cách chia'][::-1], sp.iloc[::-1, 1], color=[C['clay'], C['sand'], C['blue']])
for i, v in enumerate(sp.iloc[::-1, 1]): ax.text(v + .003, i, f'{v:.3f}', va='center')
ax.set(xlabel='RMSE (thấp hơn là tốt hơn)', title=f'Cùng một model, ba cách chia ({time.time() - t:.0f} giây)'); ax.set_xlim(0, sp.iloc[:, 1].max() * 1.15)
plt.tight_layout(); plt.show()
check('chia ngẫu nhiên cho điểm đẹp nhất', r_rand < r_day and r_rand < r_time)
note(f'Chia ngẫu nhiên báo RMSE {r_rand:.3f}; chấm đúng kiểu sử dụng (6 tháng sau) là {r_time:.3f}, tệ hơn {r_time / r_rand - 1:.0%}. '
     f'Chia theo ngày ({r_day:.3f}) đã gần sự thật hơn vì giờ kề nhau của cùng một ngày không còn "nhắc bài" cho nhau. '
     'Chọn tham số bằng cách chia ngẫu nhiên thì dễ chọn nhầm cấu hình chỉ giỏi nội suy giữa các giờ đã thấy.')

## 4. Đường tắt và rò rỉ

`hour.csv` còn hai cột `casual` (khách vãng lai) và `registered` (thành viên). Đọc Readme: `cnt = casual + registered`. Lúc cần dự đoán, hai cột này chưa có, vì chúng chính là phần của nhãn.

In [ ]:
leak_pred = cross_val_predict(gbm(), hp[FEAT + ['casual', 'registered']], yp, cv=GroupKFold(5), groups=hp['dteday'])
clean_pred = cross_val_predict(gbm(), hp[FEAT], yp, cv=GroupKFold(5), groups=hp['dteday'])
lk = pd.DataFrame([('có casual, registered', r2_score(yp, leak_pred), rmse(yp, leak_pred)), ('bỏ hai cột ấy', r2_score(yp, clean_pred), rmse(yp, clean_pred))],
                  columns=['đặc trưng', 'R²', 'RMSE'])
table(lk, 'Chia theo ngày, 18 tháng đầu', fmt={'R²': '{:.4f}', 'RMSE': '{:.3f}'})
check('casual + registered đúng bằng cnt ở mọi dòng', (h['casual'] + h['registered'] == h['cnt']).all())
note(f'Có hai cột rò rỉ, R² lên {lk.iloc[0, 1]:.4f}: model chỉ cần cộng hai số. Điểm gần hoàn hảo sau vài phút train là lúc phải đi tìm cột như thế này, '
     'bằng câu hỏi "lúc dự đoán thật, cột này đã có chưa?".')

## 5. Lệch phân phối

Học trên năm 2011 (bỏ cột `yr`), đoán năm 2012. Thời tiết hai năm giống nhau, nhưng số người thuê thì không.

In [ ]:
F_NOYR = [f for f in FEAT if f != 'yr']
y11, y12 = h['yr'] == 0, h['yr'] == 1
m11 = gbm().fit(h.loc[y11, F_NOYR], y[y11])
h12 = h[y12].assign(pred=np.expm1(m11.predict(h.loc[y12, F_NOYR])))
mon = h12.groupby('mnth')[['cnt', 'pred']].mean()
fig, ax = plt.subplots(figsize=(10, 3.6))
ax.plot(mon.index, mon['cnt'], 'o-', color=C['blue'], lw=2, label='thật (2012)')
ax.plot(mon.index, mon['pred'], 'o--', color=C['clay'], lw=2, label='model học năm 2011 đoán')
ax.set(xlabel='tháng', ylabel='lượt thuê trung bình mỗi giờ', title='Model đúng hình dạng mùa nhưng thiếu cả một bậc'); ax.legend(frameon=False); ax.set_xticks(range(1, 13))
plt.tight_layout(); plt.show()
under = 1 - h12['pred'].sum() / h12['cnt'].sum()
check('model học năm 2011 đoán thiếu năm 2012', under > 0.1)
note(f'Năm 2012 có trung bình {h12["cnt"].mean():.0f} lượt mỗi giờ, model đoán {h12["pred"].mean():.0f}: thiếu {under:.0%}. '
     'Lệch nằm ở nhãn (lượng khách tăng), không ở thời tiết, nên nhìn phân phối của từng cột đặc trưng không thấy được.')

**Adversarial validation** hỏi ngược: một classifier có phân biệt được dòng của năm 2011 với dòng của năm 2012 chỉ từ các cột đặc trưng không? AUC gần 0.5 là hai phần giống nhau; AUC gần 1 là có cột tố cáo thời điểm. Cách chia khi chấm AUC cũng phải theo ngày, nếu không classifier chỉ cần nhớ thời tiết của từng ngày.

In [ ]:
adv_y = h['yr'].values
def adv_auc(cols, cv, groups=None):
    p = cross_val_predict(HistGradientBoostingClassifier(max_iter=200, random_state=0), h[cols], adv_y, cv=cv, groups=groups, method='predict_proba')[:, 1]
    return roc_auc_score(adv_y, p)
kf, gk = KFold(5, shuffle=True, random_state=0), GroupKFold(5)
cases = [('mọi cột, kể cả instant (số thứ tự dòng)', F_NOYR + ['instant'], kf, None),
         ('bỏ instant, chia ngẫu nhiên theo giờ', F_NOYR, kf, None),
         ('bỏ instant, chia theo ngày', F_NOYR, gk, h['dteday']),
         ('bỏ thêm hum, chia theo ngày', [f for f in F_NOYR if f != 'hum'], gk, h['dteday'])]
adv = pd.DataFrame([(n, adv_auc(c, cv, g)) for n, c, cv, g in cases], columns=['cột đưa vào, cách chia', 'AUC phân biệt 2011 với 2012'])
table(adv, 'Adversarial validation', fmt={'AUC phân biệt 2011 với 2012': '{:.3f}'})
hum = h.groupby('yr')['hum'].describe()[['mean', '25%', '50%', '75%']]
table(hum.rename(index={0: '2011', 1: '2012'}).reset_index(names='năm'), 'Cột hum (độ ẩm, đã chia 100) theo năm', fmt={c: '{:.3f}' for c in hum.columns})
check('cột thứ tự dòng tố cáo ngay thời điểm', adv.iloc[0, 1] > 0.99)
note(f'Còn cột <code>instant</code> thì AUC {adv.iloc[0, 1]:.3f}: cột thứ tự dòng là đồng hồ. Bỏ nó, chia ngẫu nhiên vẫn được {adv.iloc[1, 1]:.2f} vì classifier nhớ thời tiết của từng ngày; '
     f'chia theo ngày còn {adv.iloc[2, 1]:.2f}, bỏ thêm <code>hum</code> còn {adv.iloc[3, 1]:.2f}. Cột đáng xem tiếp là cột làm AUC tụt nhiều nhất khi bỏ đi.')

## 6. Ablation: một nhóm cột có thật sự giúp không

Bỏ từng nhóm cột, train lại, chấm theo **thời gian** như mục 3. Một cột chỉ đáng giữ nếu bỏ nó làm điểm tệ đi rõ hơn mức dao động giữa các lần chạy.

In [ ]:
groups_ = {'giờ (hr)': ['hr'], 'loại ngày (weekday, workingday, holiday)': ['weekday', 'workingday', 'holiday'],
           'mùa, tháng (season, mnth)': ['season', 'mnth'], 'năm (yr)': ['yr'], 'thời tiết (weathersit, temp, atemp, hum, windspeed)': ['weathersit', 'temp', 'atemp', 'hum', 'windspeed']}
def score_time(cols, seed=0):
    m = HistGradientBoostingRegressor(max_iter=300, random_state=seed).fit(h.loc[past, cols], y[past])
    return rmse(y[future], m.predict(h.loc[future, cols]))
base_scores = [score_time(FEAT, s) for s in range(3)]
abl = pd.DataFrame([(g, score_time([f for f in FEAT if f not in cs])) for g, cs in groups_.items()], columns=['bỏ nhóm', 'RMSE'])
abl['tệ hơn'] = abl['RMSE'] - np.mean(base_scores)
abl = abl.sort_values('tệ hơn', ascending=False)
fig, ax = plt.subplots(figsize=(10, 3.4))
ax.barh(abl['bỏ nhóm'][::-1], abl['tệ hơn'][::-1], color=C['clay'])
for i, v in enumerate(abl['tệ hơn'][::-1]): ax.text(v + .005, i, f'{v:+.3f}', va='center')
ax.axvline(0, color=C['gray'], lw=1)
ax.set(xlabel='RMSE tăng thêm so với đủ cột', title=f'Đủ cột: RMSE {np.mean(base_scores):.3f} (3 seed lệch nhau tối đa {np.ptp(base_scores):.4f})')
ax.set_xlim(min(0, abl['tệ hơn'].min()) - .02, abl['tệ hơn'].max() * 1.2)
plt.tight_layout(); plt.show()
check('bỏ cột giờ làm điểm tệ đi nhiều nhất', abl.iloc[0, 0] == 'giờ (hr)')
weak = abl[abl['tệ hơn'] < 0.01]['bỏ nhóm'].tolist()
note(f'Bỏ giờ làm RMSE tăng {abl.iloc[0, 2]:+.3f}, đúng như bản đồ thứ × giờ ở mục 1. '
     + (f'Nhóm {", ".join(weak)} gần như không đổi điểm (dưới 0.01): giữ hay bỏ nên quyết bằng con số này, không bằng cảm giác. ' if weak else 'Mọi nhóm đều giúp ít nhiều. ')
     + f'Dao động giữa các seed chỉ {np.ptp(base_scores):.4f}, nên các chênh lệch lớn hơn mức ấy là thật.')

## 7. Nhìn từng lát dữ liệu

Một con số RMSE chung che mất chỗ model sai. Chia sai số trên 6 tháng chấm theo giờ trong ngày và theo thời tiết, kèm số dòng của mỗi lát.

In [ ]:
fut = h[future].assign(err=(pred_future - y[future]) ** 2)
by_hr = fut.groupby('hr')['err'].mean() ** .5
by_w = fut.groupby('weathersit').agg(n=('err', 'size'), rmse=('err', lambda e: e.mean() ** .5))
fig, axes = plt.subplots(1, 2, figsize=(15, 3.8), gridspec_kw={'width_ratios': [2, 1]})
axes[0].bar(by_hr.index, by_hr.values, color=C['blue']); axes[0].axhline(r_time, color=C['clay'], ls='--', label=f'chung: {r_time:.3f}')
axes[0].set(xlabel='giờ', ylabel='RMSE', title='Sai số theo giờ trong ngày'); axes[0].legend(frameon=False); axes[0].set_xticks(range(0, 24, 2))
names = {1: 'quang', 2: 'mây, sương', 3: 'mưa nhỏ', 4: 'mưa to'}
axes[1].bar([names[k] for k in by_w.index], by_w['rmse'], color=C['sand']); bar_labels(axes[1], '{:.2f}')
for i, n in enumerate(by_w['n']): axes[1].text(i, 0.02, f'{n} giờ', ha='center', color='white', fontsize=9)
axes[1].set(ylabel='RMSE', title='Sai số theo thời tiết')
plt.tight_layout(); plt.show()
worst_h = int(by_hr.idxmax())
note(f'Sai nhiều nhất lúc {worst_h} giờ (RMSE {by_hr.max():.2f}, gấp {by_hr.max() / r_time:.1f} lần mức chung)'
     + (': ban đêm số lượt nhỏ nên lệch vài lượt đã là lệch lớn trên thang log. ' if worst_h < 6 else '. ')
     + f'Giờ mưa nhỏ sai {by_w.loc[3, "rmse"]:.2f} so với {by_w.loc[1, "rmse"]:.2f} lúc trời quang, mà chỉ có {by_w.loc[3, "n"]} giờ: muốn cải thiện thì nhìn vào đây trước.')

## 8. Tóm tắt

In [ ]:
summary = [('giờ bị thiếu trong hour.csv (không phải ô trống)', f'{days * 24 - len(h)}'),
           ('SMS: tin trùng hệt / gần trùng', f'{dup.iloc[2, 1]} / {dup.iloc[3, 1]}'),
           ('RMSE: chia ngẫu nhiên / theo ngày / theo thời gian', f'{r_rand:.3f} / {r_day:.3f} / {r_time:.3f}'),
           ('R² khi có cột rò rỉ casual, registered', f'{lk.iloc[0, 1]:.4f}'),
           ('model học 2011 đoán thiếu 2012', f'{under:.0%}'),
           ('adversarial AUC: có instant / bỏ instant, chia theo ngày', f'{adv.iloc[0, 1]:.3f} / {adv.iloc[2, 1]:.3f}'),
           ('bỏ cột giờ: RMSE tăng', f'{abl.iloc[0, 2]:+.3f}')]
table(pd.DataFrame(summary, columns=['điều đo', 'kết quả']), 'Các con số chính của notebook')

**Thử thêm khi dạy:** ở mục 3 đổi mốc chia thời gian sang 1/1/2012; ở mục 5 thêm lại cột `yr` rồi xem model học năm 2011 còn đoán được năm 2012 không (cột `yr` chỉ có một giá trị khi train); ở mục 2 thay 1-NN bằng logistic và xem phần "đã gặp" còn quan trọng không.